In [1]:
!pip install pyarrow==14.0.1
!pip install datasets==2.18.0
!pip install transformers==4.44.0
!pip install accelerate==0.33.0
!pip install requests

In [2]:
# Import all necessary libraries

import requests
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import torch
from datasets import load_dataset, Dataset
import json
from collections import Counter
import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    hamming_loss,
    classification_report
)
import ast

In [3]:
# Test Response of URL

url = "https://api.github.com/repos/pytorch/pytorch/issues?page=1&per_page=1"
response = requests.get(url)
print(response.status_code)

200


In [4]:
print(response.json())

[{'url': 'https://api.github.com/repos/pytorch/pytorch/issues/177024', 'repository_url': 'https://api.github.com/repos/pytorch/pytorch', 'labels_url': 'https://api.github.com/repos/pytorch/pytorch/issues/177024/labels{/name}', 'comments_url': 'https://api.github.com/repos/pytorch/pytorch/issues/177024/comments', 'events_url': 'https://api.github.com/repos/pytorch/pytorch/issues/177024/events', 'html_url': 'https://github.com/pytorch/pytorch/pull/177024', 'id': 4051748409, 'node_id': 'PR_kwDOA-j9z87JYgFd', 'number': 177024, 'title': '[DOCS] Fix typo "is wrong,please" -> "is wrong, please"', 'user': {'login': 'benediktjohannes', 'id': 253604130, 'node_id': 'U_kgDODx2xIg', 'avatar_url': 'https://avatars.githubusercontent.com/u/253604130?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/benediktjohannes', 'html_url': 'https://github.com/benediktjohannes', 'followers_url': 'https://api.github.com/users/benediktjohannes/followers', 'following_url': 'https://api.github.com/users/b

In [6]:
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [7]:
# Define Function to fetch all issues

def fetch_issues(owner = "pytorch", repo = "pytorch", num_issues = 10_000, rate_limit = 10_000, issues_path = Path(".")):
    if not issues_path.is_dir():
      issues_path.mkdir(exist_ok = True)


    batch = []
    all_issues = []
    per_page = 100
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    print(f"Attempting to fetch {num_issues} issues from {owner}/{repo}")
    print(f"Number of pages to request based on num_issues: {num_pages}")
    print(f"Internal rate limit for batch processing set to: {rate_limit} issues")

    for page in tqdm(range(1, num_pages + 1)):
      query = f"issues?page={page}&per_page={per_page}&state=all"
      issues_response = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers = headers)

      print(f"--- Page {page} ---")
      print(f"Status Code: {issues_response.status_code}")

      if issues_response.status_code == 200:
          current_page_issues = issues_response.json()
          print(f"Issues received on this page: {len(current_page_issues)}")
          if not current_page_issues:
              print(f"No more issues found from page {page}. Stopping fetch.")
              break # Exit the loop if no issues are returned
          batch.extend(current_page_issues)
      else:
          print(f"Error fetching issues on page {page}. Response text: {issues_response.text}")
          break # Break on error

      if len(batch) > rate_limit and len(all_issues) < num_issues:
        all_issues.extend(batch)
        batch = []
        print(f"Reached internal rate limit ({rate_limit} issues processed). Sleeping for one hour ...")
        time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)

    # Convert 'labels' and 'body' columns to string representation
    df['labels'] = df['labels'].apply(lambda x: str(x))
    df['body'] = df['body'].apply(lambda x: str(x) if x is not None else "")

    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Total issues collected: {len(df)}. Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )


In [8]:
fetch_issues()

Attempting to fetch 10000 issues from pytorch/pytorch
Number of pages to request based on num_issues: 100
Internal rate limit for batch processing set to: 10000 issues


  0%|          | 0/100 [00:00<?, ?it/s]

--- Page 1 ---
Status Code: 200
Issues received on this page: 100
--- Page 2 ---
Status Code: 200
Issues received on this page: 100
--- Page 3 ---
Status Code: 200
Issues received on this page: 100
--- Page 4 ---
Status Code: 200
Issues received on this page: 100
--- Page 5 ---
Status Code: 200
Issues received on this page: 100
--- Page 6 ---
Status Code: 200
Issues received on this page: 100
--- Page 7 ---
Status Code: 200
Issues received on this page: 100
--- Page 8 ---
Status Code: 200
Issues received on this page: 100
--- Page 9 ---
Status Code: 200
Issues received on this page: 100
--- Page 10 ---
Status Code: 200
Issues received on this page: 100
--- Page 11 ---
Status Code: 200
Issues received on this page: 100
--- Page 12 ---
Status Code: 200
Issues received on this page: 100
--- Page 13 ---
Status Code: 200
Issues received on this page: 100
--- Page 14 ---
Status Code: 200
Issues received on this page: 100
--- Page 15 ---
Status Code: 200
Issues received on this page: 100
--- 

In [9]:
df = pd.read_json("pytorch-issues.jsonl", lines=True)
print(len(df))

9900


In [10]:
# Load the existing file
df = pd.read_json("pytorch-issues.jsonl", lines=True)

# Parse the stringified labels and extract names
def parse_and_extract_labels(label_str):
    try:

        label_dicts = ast.literal_eval(label_str)

        return [label['name'] for label in label_dicts if isinstance(label, dict)]
    except:
        return []

df['labels'] = df['labels'].apply(parse_and_extract_labels)

# Convert to HF dataset
dataset = Dataset.from_pandas(df)


from collections import Counter
all_labels = []
for issue in dataset:
    all_labels.extend(issue['labels'])

label_counts = Counter(all_labels)
print(f"Total unique labels: {len(label_counts)}")
print(f"Most common: {label_counts.most_common(20)}")

Total unique labels: 404
Most common: [('topic: not user facing', 4196), ('Merged', 3805), ('ciflow/trunk', 3529), ('triaged', 3406), ('open source', 3268), ('ciflow/inductor', 3087), ('module: inductor', 2247), ('module: dynamo', 1427), ('module: rocm', 1189), ('oncall: pt2', 942), ('ci-no-td', 640), ('fb-exported', 568), ('meta-exported', 568), ('rocm-skipped-tests', 520), ('Stale', 495), ('bot-triaged', 488), ('Reverted', 439), ('ciflow/rocm-mi300', 355), ('release notes: distributed (dtensor)', 318), ('ciflow/b200', 272)]


In [11]:
min_label_count = 100  # Adjust based on your data

# Filter labels
valid_labels = [label for label, count in label_counts.items()
                if count >= min_label_count]
valid_labels.sort()

print(f"\nUsing {len(valid_labels)} labels (out of {len(label_counts)})")
print(f"Minimum count threshold: {min_label_count}")

# Create mappings
label2id = {label: idx for idx, label in enumerate(valid_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(valid_labels)


print("\nFinal label set:")
for label in valid_labels:
    print(f"  {label:40s} (count: {label_counts[label]})")


Using 45 labels (out of 404)
Minimum count threshold: 100

Final label set:
  Merged                                   (count: 3805)
  Reverted                                 (count: 439)
  Stale                                    (count: 495)
  bot-triaged                              (count: 488)
  ci-no-td                                 (count: 640)
  ciflow/b200                              (count: 272)
  ciflow/h100                              (count: 202)
  ciflow/inductor                          (count: 3087)
  ciflow/inductor-pallas                   (count: 108)
  ciflow/mps                               (count: 159)
  ciflow/rocm                              (count: 156)
  ciflow/rocm-mi300                        (count: 355)
  ciflow/trunk                             (count: 3529)
  ciflow/xpu                               (count: 163)
  fb-exported                              (count: 568)
  fx                                       (count: 153)
  high priority         

In [12]:
# Keep only issues that have at least one valid label
def has_valid_labels(example):
    return any(label in label2id for label in example['labels'])

filtered_dataset = dataset.filter(has_valid_labels)

print(f"\nDataset filtering:")
print(f"  Original size: {len(dataset)}")
print(f"  Filtered size: {len(filtered_dataset)}")
print(f"  Removed: {len(dataset) - len(filtered_dataset)}")

Filter:   0%|          | 0/9900 [00:00<?, ? examples/s]


Dataset filtering:
  Original size: 9900
  Filtered size: 9609
  Removed: 291


In [13]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    """
    Preprocess PyTorch GitHub issues for multi-label classification
    """

    # Tokenize
    tokenized = tokenizer(
        examples['body'],
        truncation=True,
        return_tensors=None
    )

    # Convert labels to binary vectors
    binary_labels = []
    for issue_labels in examples['labels']:

        binary_vector = [0.0] * num_labels

        # Set 1.0 for each valid label present
        for label in issue_labels:
            if label in label2id:
                idx = label2id[label]
                binary_vector[idx] = 1.0

        binary_labels.append(binary_vector)

    tokenized['labels'] = binary_labels

    return tokenized

# Test on a single example first
print("\n--- Testing preprocessing on one example ---")
test_example = filtered_dataset[0]
print(f"Original labels: {test_example['labels']}")

test_processed = preprocess_function({
    'title': [test_example['title']],
    'body': [test_example['body']],
    'labels': [test_example['labels']]
})

print(f"Binary vector: {test_processed['labels'][0]}")
print(f"Which labels are 1: {[id2label[i] for i, val in enumerate(test_processed['labels'][0]) if val == 1.0]}")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


--- Testing preprocessing on one example ---
Original labels: ['open source', 'topic: not user facing']
Binary vector: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
Which labels are 1: ['open source', 'topic: not user facing']


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [14]:
# Map the tokenization function to the filtered dataset

tokenized_dataset = filtered_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000,
    remove_columns=filtered_dataset.column_names,
    desc="Tokenizing dataset"
)

print(f"\nTokenized dataset:")
print(tokenized_dataset)
print(f"\nExample processed item keys: {tokenized_dataset[0].keys()}")
print(f"Input IDs shape: {len(tokenized_dataset[0]['input_ids'])}")
print(f"Labels shape: {len(tokenized_dataset[0]['labels'])}")

Tokenizing dataset:   0%|          | 0/9609 [00:00<?, ? examples/s]


Tokenized dataset:
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9609
})

Example processed item keys: dict_keys(['labels', 'input_ids', 'attention_mask'])
Input IDs shape: 19
Labels shape: 45


In [15]:
# Create Train and Test Split

tokenized_dataset = tokenized_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

print(f"\nDataset splits:")
print(f"  Train: {len(tokenized_dataset['train'])} examples") # For Training
print(f"  Test:  {len(tokenized_dataset['test'])} examples") # For Eval


tokenized_dataset.save_to_disk("./pytorch-issues-processed")


Dataset splits:
  Train: 7687 examples
  Test:  1922 examples


Saving the dataset (0/1 shards):   0%|          | 0/7687 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1922 [00:00<?, ? examples/s]

In [16]:
# Define Data Collator

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [17]:
def compute_metrics(eval_pred):
    """
    Compute multi-label classification metrics
    """
    logits, labels = eval_pred

    predictions = torch.sigmoid(torch.tensor(logits)).numpy()

    # Apply threshold to get binary predictions
    threshold = 0.5
    binary_predictions = (predictions > threshold).astype(int)


    precision_micro = precision_score(labels, binary_predictions, average='micro', zero_division=0)
    recall_micro = recall_score(labels, binary_predictions, average='micro', zero_division=0)
    f1_micro = f1_score(labels, binary_predictions, average='micro', zero_division=0)


    precision_macro = precision_score(labels, binary_predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, binary_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, binary_predictions, average='macro', zero_division=0)


    f1_weighted = f1_score(labels, binary_predictions, average='weighted', zero_division=0)


    f1_samples = f1_score(labels, binary_predictions, average='samples', zero_division=0)


    subset_accuracy = accuracy_score(labels, binary_predictions)


    hamming = hamming_loss(labels, binary_predictions)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'f1_samples': f1_samples,
        'precision_micro': precision_micro,
        'precision_macro': precision_macro,
        'recall_micro': recall_micro,
        'recall_macro': recall_macro,
        'subset_accuracy': subset_accuracy,
        'hamming_loss': hamming,
    }

In [18]:
model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# Define Training Arguments

training_args = TrainingArguments(

    output_dir=f"{model_name}-pytorch-issues-classifier",

    # Evaluation
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,

    # Training hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,

    # Optimization
    warmup_steps=500,

    # Logging
    logging_dir='./logs',
    logging_steps=100,

    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    save_total_limit=2,
    fp16=True,
    report_to="none",
)

In [20]:
# Define Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator = data_collator
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [21]:
trainer.train()

Step,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted,F1 Samples,Precision Micro,Precision Macro,Recall Micro,Recall Macro,Subset Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
500,0.215700,0.200676,0.376279,0.064290,0.285564,0.303944,0.662933,0.094182,0.262691,0.058334,0.026015,0.072598,11.559700,166.268000,10.467000
1000,0.162200,0.153752,0.590954,0.168055,0.513690,0.532398,0.725859,0.244527,0.498336,0.157542,0.078044,0.057510,12.082200,159.078000,10.015000
1500,0.137600,0.137232,0.666310,0.245265,0.601896,0.620368,0.741667,0.260229,0.604854,0.239296,0.169095,0.050503,12.225700,157.209000,9.897000
2000,0.131600,0.128754,0.687656,0.259927,0.620871,0.638834,0.756788,0.262548,0.630097,0.258880,0.181061,0.047716,12.249000,156.910000,9.878000
2500,0.122600,0.124954,0.695976,0.262902,0.628728,0.649221,0.768960,0.265220,0.635645,0.261896,0.188345,0.046294,12.189200,157.680000,9.927000


TrainOutput(global_step=2883, training_loss=0.1822233123991004, metrics={'train_runtime': 584.0366, 'train_samples_per_second': 39.486, 'train_steps_per_second': 4.936, 'total_flos': 2885540446171980.0, 'train_loss': 0.1822233123991004, 'epoch': 3.0})

In [22]:
from huggingface_hub import notebook_login

notebook_login()

In [23]:
trainer.push_to_hub(commit_message="Training complete", tags="label-classification")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ssifier/model.safetensors:   0%|          | 35.1kB /  268MB            

  ...ssifier/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

CommitInfo(commit_url='https://huggingface.co/yajatpawar/distilbert-base-uncased-pytorch-issues-classifier/commit/ab634e2169a18b22386dcc15481ceed8d8cbb119', commit_message='Training complete', commit_description='', oid='ab634e2169a18b22386dcc15481ceed8d8cbb119', pr_url=None, repo_url=RepoUrl('https://huggingface.co/yajatpawar/distilbert-base-uncased-pytorch-issues-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='yajatpawar/distilbert-base-uncased-pytorch-issues-classifier'), pr_revision=None, pr_num=None)

In [24]:
tokenized_dataset.push_to_hub("yajatpawar/pytorch-issues-dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/yajatpawar/pytorch-issues-dataset/commit/e0fedd0012507eff9b6d1e11dd07a1ad6cf1649d', commit_message='Upload dataset', commit_description='', oid='e0fedd0012507eff9b6d1e11dd07a1ad6cf1649d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/yajatpawar/pytorch-issues-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='yajatpawar/pytorch-issues-dataset'), pr_revision=None, pr_num=None)

In [30]:
filtered_dataset.push_to_hub("yajatpawar/pytorch-issues-dataset-clean")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/yajatpawar/pytorch-issues-dataset-clean/commit/a6b87f880c77afe9c5e020378464b8fc9a484442', commit_message='Upload dataset', commit_description='', oid='a6b87f880c77afe9c5e020378464b8fc9a484442', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/yajatpawar/pytorch-issues-dataset-clean', endpoint='https://huggingface.co', repo_type='dataset', repo_id='yajatpawar/pytorch-issues-dataset-clean'), pr_revision=None, pr_num=None)

In [25]:
from huggingface_hub import HfApi
import json

# Create label mappings file
label_mappings = {
    'label2id': label2id,
    'id2label': id2label,
    'num_labels': num_labels,
    'valid_labels': valid_labels
}


with open('label_mappings.json', 'w') as f:
    json.dump(label_mappings, f, indent=2)

print("✅ Saved label_mappings.json locally")

# Upload to the model repo
api = HfApi()
api.upload_file(
    path_or_fileobj="label_mappings.json",
    path_in_repo="label_mappings.json",
    repo_id="yajatpawar/distilbert-base-uncased-pytorch-issues-classifier",
    repo_type="model"
)

print("✅ Uploaded label_mappings.json to model repo!")

✅ Saved label_mappings.json locally
✅ Uploaded label_mappings.json to model repo!


In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.push_to_hub("yajatpawar/distilbert-base-uncased-pytorch-issues-classifier")

print("✅ Done!")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Done!


In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download
import json

# Load model
model_name = "yajatpawar/distilbert-base-uncased-pytorch-issues-classifier"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load label mappings
label_mappings_path = hf_hub_download(repo_id=model_name, filename="label_mappings.json")
with open(label_mappings_path, 'r') as f:
    label_info = json.load(f)
    id2label = {int(k): v for k, v in label_info['id2label'].items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_top_k(body, k=3):
    """
    Return top K most likely labels regardless of threshold
    """
    text = body if body else ""
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    # Get top K
    top_indices = np.argsort(probs)[-k:][::-1]
    predictions = [(id2label[i], float(probs[i])) for i in top_indices]

    return predictions

# Test
top_3 = predict_top_k(test_body, k=3)
print("=== Top 3 Predictions ===")
for label, prob in top_3:
    print(f"{label:40s} {prob:.4f}")

=== Top 3 Predictions ===
triaged                                  0.4915
open source                              0.4540
topic: not user facing                   0.2954
